## Deep Research

Enhance Deep Research workflow with clarifying questions and report evaluation.

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool, OpenAIChatCompletionsModel
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
from openai import AsyncOpenAI

In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
sendgrid_api_key = os.getenv("SENDGRID_API_KEY")
openrouter_url = "https://openrouter.ai/api/v1"
email_address = os.getenv("EMAIL_ADDRESS")

In [ ]:
client = AsyncOpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

model = OpenAIChatCompletionsModel(model="openai/gpt-4o-mini", openai_client=client)
model_nano = OpenAIChatCompletionsModel(model="openai/gpt-4.1-nano", openai_client=client)

### Search Agent

In [ ]:
INSTRUCTIONS = (
    "You are a research assistant. Given a search term, you search the web for that term and "
    "produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 "
    "words. Capture the main points. Write succintly, no need to have complete sentences or good "
    "grammar. This will be consumed by someone synthesizing a report, so its vital you capture the "
    "essence and ignore any fluff. Do not include any additional commentary other than the summary itself."
)

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

### Search Planning Agent

In [ ]:


HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."


class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")
    
planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=model,
    output_type=WebSearchPlan,
)

### Email Agent

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(email_address)  
    to_email = To(email_address)  
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print("Email response", response.status_code)
    return "success"


INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=model,
)

### Writer Agent

In [ ]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model=model,
    output_type=ReportData,
)

### Additional Context Agent

In [ ]:
INSTRUCTIONS = f"You are a helpful research assistant. Given a query, generate three questions that can be used to \
get additional context for the query. Output list of questions."

class FollowUpQuestions(BaseModel):
    questions: list[str] = Field(description="A list of three questions that can be used to get additional context for the query.")

followup_agent = Agent(
    name="FollowUpAgent",
    instructions=INSTRUCTIONS,
    model=model,
    output_type=FollowUpQuestions,
)

### Evaluator Agent

In [ ]:
INSTRUCTIONS = (
    "You an evaluator. Your task is to evaluate the quality of a research report. Consider the following criteria:\n\n"
    "1. Accuracy: Are the claims in the report supported by evidence? Are there any factual inaccuracies?\n"
    "2. Completeness: Does the report cover all relevant aspects of the topic? Are there any important points missing?\n"
    "3. Clarity: Is the report well-organized and easy to understand? Is the writing clear and concise?\n"
    "4. Insightfulness: Does the report provide valuable insights or analysis on the topic? Does it go beyond surface-level information?\n"
    "5. Usefulness: How useful is the report for someone looking to understand the topic? Does it provide actionable insights or recommendations?\n\n"

    "If the report is acceptable, return is_acceptable as true and do not provide any feedback. If the report is not acceptable, return is_acceptable as false and provide specific feedback on what needs improvement and which criteria are not met."
)
class Evaluation(BaseModel):
    is_acceptable: bool = Field(description="Whether the report is acceptable based on the evaluation criteria.")
    feedback: str = Field(description="Feedback on what needs improvement if the report is not acceptable.")

evaluation_agent = Agent(
    name="EvaluationAgent",
    instructions=INSTRUCTIONS,
    model=model,
    output_type=Evaluation,
)

### The next 4 functions will plan and execute the search, using planner_agent, search_agent and followup_agent

In [ ]:
async def generate_followup_questions(query: str):
    """ Use the followup_agent to generate follow up questions for the query """
    print("Generating follow up questions...")
    result = await Runner.run(followup_agent, f"Query: {query}")
    print("Follow up questions:")
    context = ""
    for question in result.final_output.questions:
        print(f"- {question}")
        context += f"- {question}:\n"
        answer = input(f"{question} (or leave blank to skip): ")
        context += f"  - Answer: {answer}\n"

    new_query = f"Original query: {query}\n\n Below is additional context from the follow-up questions:\n{context}\n\nUse this additional context to help you answer the original query as best as possible."

    return new_query


async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 3 functions write a report, evaluate and email it

In [ ]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    writer_input = (
        f"Original query: {query}\n"
        f"Summarized search results: {search_results}"
    )
    result = await Runner.run(writer_agent, writer_input)

    MAX_REVISIONS = 1
    for revision_num in range(MAX_REVISIONS):
        evaluation = await evaluate_report(query, result.final_output.markdown_report)
        
        if evaluation.is_acceptable:
            print(f"Report accepted after {revision_num} revision(s).")
            break

        print(f"Revision {revision_num + 1}/{MAX_REVISIONS} - Feedback:\n{evaluation.feedback}")

        revised_input = (
            f"Original query: {query}\n"
            f"Original search results: {search_results}\n\n"
            f"Current report:\n{result.final_output.markdown_report}\n\n"
            f"Feedback to address:\n{evaluation.feedback}\n\n"
            "Please revise the report to address the feedback while staying true to the original query."
        )
        result = await Runner.run(writer_agent, revised_input)
    else:
        print(f"Warning: Report was not accepted after {MAX_REVISIONS} revision(s). Returning best attempt.")

    print("Finished writing report.")
    return result.final_output


async def evaluate_report(query: str, report: str):
    """Use the evaluation agent to evaluate the report and return its output."""
    print("Evaluating report...")
    
    evaluation_input = (
        f"Original query: {query}\n\n"
        f"Report:\n{report}\n\n"
        "Evaluate the report based on the criteria and provide feedback if it is not acceptable."
    )
    evaluation = await Runner.run(evaluation_agent, evaluation_input)

    if evaluation.final_output.is_acceptable:
        print("Report is acceptable.")
    else:
        print(f"Report is not acceptable. Feedback:\n{evaluation.final_output.feedback}")

    return evaluation.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Running the workflow

In [ ]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Generating follow up questions...")
    query = await generate_followup_questions(query)
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")


